[Guide Index](README.md)

# 04.01 SRL (SPARQL Rules)

SRL (SPARQL Rules) is a rules language for deriving new triples from the ones in the graph. A rule says "when this pattern matches, add these triples", and StarLayer runs a set of rules over a graph. It is a Datalog-style language, independent of SHACL, specified by the W3C as SPARQL-RL, [SPARQL 1.2 RL](https://www.w3.org/TR/sparql12-rl/).

A rule set is parsed from text with `srl.parse_ruleset()`. Then `ruleset.infer(graph)` returns the new triples as a graph, and `ruleset.query(graph, goal)` answers a question. The graph can be a `StarLayerGraph` or a `StarLayerDataset`.

This guide builds rules up one feature at a time. [04.02](04.02-srl-running-rule-sets.ipynb) explains how a rule set runs, which rule sets are rejected and why; [04.03](04.03-srl-more-features.ipynb) covers more features: triple terms, property paths and aggregation; and [04.04](04.04-srl-vs-sparql.ipynb) compares SRL with SPARQL.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import NamedNode, StarLayerGraph
from starlayer.sparql import srl

EX = "http://example.org/"


def ex(name):
    """A term in the example namespace, as a pyoxigraph NamedNode."""
    return NamedNode(EX + name)


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses.
    (A new StarLayerGraph has no prefixes; the ones used here are bound explicitly.)"""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 04.01.1 The base graph

The examples use a small family graph: Alice is Bob's parent, Bob is Carol's parent, and Dan has no family in the graph. Everyone has an age.

In [2]:
base = StarLayerGraph(prefixes={"ex": EX})
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:alice ex:parentOf ex:bob ;   ex:age 52 .
    ex:bob   ex:parentOf ex:carol ; ex:age 30 .
    ex:carol ex:age 7 .
    ex:dan   ex:age 41 .
""", format="turtle")

print("the base graph")
show(base)

the base graph
@prefix ex: <http://example.org/> .

ex:alice ex:age 52 ;
    ex:parentOf ex:bob .

ex:bob ex:age 30 ;
    ex:parentOf ex:carol .

ex:carol ex:age 7 .

ex:dan ex:age 41 .



## 04.01.2 A first rule

A rule has the form `RULE { head } WHERE { body }`. The `WHERE` pattern, with limited differences, is a standard SPARQL WHERE clause matched against the graph.  For every match the head says which triples to create. A triple is only created if it is not already in the graph.  

This rule says: if `?p` is the parent of `?c`, then `?c` is a child of `?p`.

`srl.parse_ruleset()` parses SRL text into a rule set. `ruleset.infer(graph)` runs it and returns a new graph that holds only the triples the rules derived. The graph passeed in is not changed.

In [3]:
child_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?c ex:childOf ?p } 
WHERE { ?p ex:parentOf ?c }
""")
print(f"{len(child_rules.rules)} rule parsed")

# inferred holds only the new triples
inferred = child_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)
print("the base graph still has", len(base), "triples")

1 rule parsed
@prefix ex: <http://example.org/> .

ex:bob ex:childOf ex:alice .

ex:carol ex:childOf ex:bob .

the base graph still has 6 triples


## 04.01.3 Matching more than one pattern

A rule body can have several patterns. The rule fires for each way of matching all of them with the same values for the variables (a join). Here a grandparent is a parent's parent:

In [4]:
grandparent_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?a ex:grandparentOf ?c } 
WHERE { ?a ex:parentOf ?b . 
?b ex:parentOf ?c }
""")

# inferred is the set of new triples.
inferred = grandparent_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:alice ex:grandparentOf ex:carol .



## 04.01.4 Rules that build on each other

A rule can match triples that other rules create. The two rules below find ancestors at any distance. R1 says a parent is an ancestor. R2 says that an ancestor of a parent is an ancestor, so it matches its own output. Both rules have the same head: several rules can have the same head, and each is another way to derive the same triple(s).

SRL runs the rules in repeated passes and stops when a pass produces nothing new. Here the first pass finds Alice as Bob's ancestor and Bob as Carol's. The second finds Alice as Carol's ancestor, and a third pass finds nothing more.

In [5]:
ancestor_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

# R1: a parent is an ancestor.
RULE { ?a ex:ancestorOf ?b } WHERE { ?a ex:parentOf ?b }

# R2: an ancestor of a parent is an ancestor (recursive - it matches its own output).
RULE { ?a ex:ancestorOf ?c } WHERE { ?a ex:parentOf ?b . ?b ex:ancestorOf ?c }
""")

inferred = ancestor_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:alice ex:ancestorOf ex:bob, ex:carol .

ex:bob ex:ancestorOf ex:carol .



## 04.01.5 `FILTER`


A `FILTER` works as it does in a SPARQL query, and most SPARQL expressions and functions can be used. There are a few exceptions outlined in [04.04](04.04-srl-vs-sparql.ipynb).

In [6]:
adult_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x a ex:Adult } WHERE { ?x ex:age ?age . FILTER(?age >= 18) }
""")

inferred = adult_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:alice a ex:Adult .

ex:bob a ex:Adult .

ex:dan a ex:Adult .



## 04.01.6 Negation with `NOT`

`NOT { }` is true when the pattern inside it has no match in the graph. This rule finds people aged 18 or more who have no children. `?x ex:age ?age` binds `?x` to the people to test, because a `NOT` only filters and cannot bind a variable itself. 

If another rule creates the triples a `NOT` looks at, SRL runs that rule first.  Otherwise `NOT { }` could be true on a first pass and false on subsequent passes.  The negation needs to see the results after all prior passes. [04.02](04.02-srl-running-rule-sets.ipynb) explains the order.

In [7]:
childless_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:childless true } WHERE { ?x ex:age ?age . FILTER(?age >= 18) NOT { ?x ex:parentOf ?child } }
""")

inferred = childless_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:dan ex:childless true .



## 04.01.7 Computing values with `SET`

A rule can compute a value with `SET(?var := expression)` in its body, using the standard SPARQL expressions. SET is SRL's counterpart of SPARQL's `BIND`. The critical difference is an expression that fails (e.g dividing by zero): `BIND` keeps the match and leaves the variable unbound, while `SET` drops the match.

In [8]:
next_year_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:ageNextYear ?n } WHERE { ?x ex:age ?a . SET(?n := ?a + 1) }
""")

inferred = next_year_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:alice ex:ageNextYear 53 .

ex:bob ex:ageNextYear 31 .

ex:carol ex:ageNextYear 8 .

ex:dan ex:ageNextYear 42 .



## 04.01.8 New nodes in a rule head

A rule creates a node by writing it in the head. How many nodes it creates depends on what it writes:

- **A fixed IRI** is one node, shared by every match. `ex:Adult` in section 04.01.5 is an example.
- **A variable IRI**, built with `SET`, can be a different node for each match.
- **A blank node** is a new node for every match.

Every parent below gets a mark `ex:Parent` (a fixed IRI), a named record (a variable IRI) and a record of their own with no name (a blank node):

In [9]:
record_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x a ex:Parent } 
WHERE { ?x ex:parentOf ?child }

RULE { ?x ex:namedRecord ?r . ?r ex:about ?x }
WHERE { ?x ex:parentOf ?child . SET(?r := IRI(CONCAT("http://example.org/record-", STRAFTER(STR(?x), "http://example.org/")))) }

RULE { ?x ex:record _:r . _:r ex:about ?x } 
WHERE { ?x ex:parentOf ?child }
""")

inferred = record_rules.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:alice a ex:Parent ;
    ex:namedRecord ex:record-alice ;
    ex:record [
        ex:about ex:alice
    ] .

ex:bob a ex:Parent ;
    ex:namedRecord ex:record-bob ;
    ex:record [
        ex:about ex:bob
    ] .

ex:record-alice ex:about ex:alice .

ex:record-bob ex:about ex:bob .



## 04.01.9 Asking a question with `query()`

`infer()` returns a graph of everything the rules derive. `ruleset.query(base_graph, goal)` is used to answer a specific question. It runs the rules the goal depends on, then matches the *goal*, a single triple pattern written in SPARQL form, against the union of the base graph with the inferred triples. It returns a list with one dictionary for each match, from the goal's variable to its value. The prefixes used are the graph's own, and then the rule set's. 

Here the question is: who are Carol's ancestors?

In [10]:
# The goal is one triple pattern: which ?who are the ancestors of Carol?
goal = "?who ex:ancestorOf ex:carol"

# query() returns one dictionary for each match, from the goal's variable to its value
rows = ancestor_rules.query(base, goal)
print(len(rows), "matches:", sorted(base.qname_term(term) for row in rows for term in row.values()))

2 matches: ['ex:alice', 'ex:bob']


A goal with no variables is a yes-or-no question: the list has one empty dictionary when the triple holds, and is empty when it does not.

In [11]:
# Is Alice an ancestor of Carol? Is Carol an ancestor of Alice?
print(bool(ancestor_rules.query(base, "ex:alice ex:ancestorOf ex:carol")))
print(bool(ancestor_rules.query(base, "ex:carol ex:ancestorOf ex:alice")))

True
False
